In [35]:
%pip install pandas numpy plotly nbformat scikit-learn ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [36]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import kmeans as kms
import hierarchical_clustering as hclust
import dbscan as dbscan

from sklearn.datasets import make_blobs

In [37]:
N_SAMPLES = 300
N_FEATURES = 2
N_CLUSTERS = 4

In [38]:
X, y_true = make_blobs(
    n_samples=N_SAMPLES,
    centers=N_CLUSTERS,
    n_features=N_FEATURES,
    random_state=42,
    cluster_std=0.7
)

data = pd.DataFrame(X, columns=[f'Feature {i+1}' for i in range(N_FEATURES)])


In [39]:
# Visualize the generated data
fig = px.scatter(
    data,
    x='Feature 1',
    y='Feature 2',
    title='Сгенерированные данные для кластеризации',
    labels={'Feature 1': 'Признак 1', 'Feature 2': 'Признак 2'},
    color=y_true.astype(str),
    category_orders={'color': [str(i) for i in range(N_CLUSTERS)]},
    hover_name=data.index,
    size_max=10
)

fig.update_layout(
    width=800,
    height=600,
    font=dict(size=12)
)

fig.show()

# Функции визаулизации

In [40]:
def visualize_clusters(data : pd.DataFrame, clusters : np.ndarray, title : str) -> None:
    data_with_clusters = data.copy()
    data_with_clusters['Cluster'] = clusters.astype(str)

    fig = px.scatter(
        data_with_clusters,
        x='Feature 1',
        y='Feature 2',
        color='Cluster',
        title=title,
        labels={'Feature 1': 'Признак 1', 'Feature 2': 'Признак 2'},
        hover_name=data_with_clusters.index,
        size_max=10
    )

    fig.update_layout(
        width=800,
        height=600,
        font=dict(size=12)
    )

    fig.show()

# Метод K-means

In [41]:
elbow_trainer = kms.ElbowTrainerImpl(
    X=X,
    ks=range(1, 11),
    km_stopper=kms.KMeansIterationStoperItersCount(100),
)

elbow_results = elbow_trainer.train()

fig_elbow = go.Figure()

fig_elbow.add_trace(
    go.Scatter(
        x=[res['k'] for res in elbow_results],
        y=[res['inertia'] for res in elbow_results],
        mode='lines+markers',
        name='Инерция',
        marker=dict(size=8),
        line=dict(width=2)
    )
)

fig_elbow.update_layout(
    title='Метод локтя для выбора оптимального количества кластеров',
    xaxis_title='Количество кластеров (k)',
    yaxis_title='Инертия',
    width=800,
    height=600,
    hovermode='x unified',
    font=dict(size=12)
)

fig_elbow.show()


In [42]:
OPTIMAL_CLUSTERS = 4

In [43]:

kmeans_final = kms.KMeansImpl(
    X=X,
    clusters_count=OPTIMAL_CLUSTERS,  
    stopper=kms.KMeansIterationStoperItersCount(100),
)
kmeans_final.fit()

clusters = np.array([kmeans_final.predict(x) for x in X])

centers = kmeans_final.centers

In [44]:
visualize_clusters(data, clusters, f'Результаты K-means кластеризации (k={OPTIMAL_CLUSTERS})')

# Иерархическая кластеризация

## Агломеративный метод

In [45]:
agglomerative : hclust.HierarchicalClustering = hclust.AgglomerativeClustering(
    X=X,
    linkage_method=hclust.AverageLinkage(),
    distance_metric=hclust.EuclideanDistance()
)

agglomerative.fit()

clusters = agglomerative.predict(OPTIMAL_CLUSTERS)


In [46]:
visualize_clusters(
    data,
    clusters,
    f'Результаты Агломеративной кластеризации (k={OPTIMAL_CLUSTERS})'
)

In [47]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider

def get_dendrogram_coords(node, x=0):
    """
    Вычисляет координаты для визуализации дендрограммы
    
    Args:
        node: узел дендрограммы
        x: текущая x-координата
    
    Returns:
        coords: словарь с координатами линий
        next_x: следующая доступная x-координата
        leaf_positions: позиции листьев
    """
    if node.left_child is None and node.right_child is None:
        # Листовой узел
        return {'x': x, 'y': 0}, x + 1, {node.index: x}
    
    # Рекурсивно обрабатываем потомков
    left_coords, x_after_left, left_positions = get_dendrogram_coords(node.left_child, x)
    right_coords, x_after_right, right_positions = get_dendrogram_coords(node.right_child, x_after_left)
    
    # Позиция текущего узла - между потомками
    current_x = (left_coords['x'] + right_coords['x']) / 2
    current_y = node.distance
    
    # Объединяем позиции листьев
    leaf_positions = {**left_positions, **right_positions}
    
    return {
        'x': current_x,
        'y': current_y,
        'left': left_coords,
        'right': right_coords
    }, x_after_right, leaf_positions

def draw_dendrogram_lines(node, coords, x_lines, y_lines):
    """
    Рекурсивно строит линии дендрограммы
    
    Args:
        node: узел дендрограммы
        coords: координаты узла
        x_lines: список x-координат линий
        y_lines: список y-координат линий
    """
    if 'left' not in coords:
        return
    
    # Линия к левому потомку
    x_lines.extend([coords['x'], coords['left']['x'], coords['left']['x'], None])
    y_lines.extend([coords['y'], coords['y'], coords['left']['y'], None])
    
    # Линия к правому потомку
    x_lines.extend([coords['x'], coords['right']['x'], coords['right']['x'], None])
    y_lines.extend([coords['y'], coords['y'], coords['right']['y'], None])
    
    # Рекурсивно обрабатываем потомков
    draw_dendrogram_lines(node.left_child, coords['left'], x_lines, y_lines)
    draw_dendrogram_lines(node.right_child, coords['right'], x_lines, y_lines)

def visualize_dendrogram_with_slider(clustering_model : hclust.HierarchicalClustering, X, data):
    """
    Визуализирует дендрограмму с интерактивным слайдером для выбора количества кластеров
    
    Args:
        clustering_model: модель кластеризации (агломеративная или дивизивная)
        X: данные
        data: DataFrame с данными
    """
    dendrogram = clustering_model.get_dendrogram()
    coords, _, _leaf_positions = get_dendrogram_coords(dendrogram)
    
    # Создаем интерактивную функцию
    def update_plot(n_clusters):
        # Получаем кластеры для выбранного числа
        clusters = clustering_model.predict(n_clusters)
        
        # Создаем subplot с двумя графиками
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                f'Разбиение на {n_clusters} кластеров'
            ),
            horizontal_spacing=0.15
        )
        
        # Строим дендрограмму
        x_lines, y_lines = [], []
        draw_dendrogram_lines(dendrogram, coords, x_lines, y_lines)
        
        fig.add_trace(
            go.Scatter(
                x=x_lines,
                y=y_lines,
                mode='lines',
                line=dict(color='blue', width=2),
                showlegend=False,
                hoverinfo='skip'
            ),
            row=1, col=1
        )
        
        # Добавляем горизонтальную линию разреза
        if n_clusters > 1 and n_clusters < len(X):
            # Вычисляем порог разреза
            if hasattr(clustering_model, 'split_history'):
                # Дивизивный метод
                split_distances = sorted([s['distance'] for s in clustering_model.split_history], reverse=True)
                if n_clusters - 1 < len(split_distances):
                    if n_clusters < len(split_distances):
                        threshold = (split_distances[n_clusters - 1] + split_distances[n_clusters]) / 2.0
                    else:
                        threshold = split_distances[n_clusters - 1] - 0.001
                else:
                    threshold = 0
            else:
                # Агломеративный метод
                n_samples = len(X)
                merge_distances = sorted([node.distance for node in clustering_model.nodes[n_samples:] 
                                        if node.left_child is not None and node.right_child is not None])
                threshold_idx = len(merge_distances) - n_clusters
                if threshold_idx >= 0 and threshold_idx < len(merge_distances):
                    if threshold_idx + 1 < len(merge_distances):
                        threshold = (merge_distances[threshold_idx] + merge_distances[threshold_idx + 1]) / 2.0
                    else:
                        threshold = merge_distances[threshold_idx] + 0.001
                else:
                    threshold = 0
            
            x_values = [x for x in x_lines if x is not None]
            fig.add_trace(
                go.Scatter(
                    x=[min(x_values), max(x_values)],
                    y=[threshold, threshold],
                    mode='lines',
                    line=dict(color='red', width=2, dash='dash'),
                    name='Уровень разреза',
                    showlegend=True
                ),
                row=1, col=1
            )
        
        # Строим scatter plot с кластерами
        data_with_clusters = data.copy()
        data_with_clusters['Cluster'] = clusters.astype(str)
        
        # Создаем scatter plot для каждого кластера
        for cluster_id in sorted(data_with_clusters['Cluster'].unique()):
            cluster_data = data_with_clusters[data_with_clusters['Cluster'] == cluster_id]
            fig.add_trace(
                go.Scatter(
                    x=cluster_data['Feature 1'],
                    y=cluster_data['Feature 2'],
                    mode='markers',
                    name=f'Кластер {cluster_id}',
                    marker=dict(size=8),
                    showlegend=True
                ),
                row=1, col=2
            )
        
        # Настройка осей
        fig.update_xaxes(title_text="Позиция", row=1, col=1)
        fig.update_yaxes(title_text="Расстояние", row=1, col=1)
        fig.update_xaxes(title_text="Признак 1", row=1, col=2)
        fig.update_yaxes(title_text="Признак 2", row=1, col=2)
        
        fig.update_layout(
            height=500,
            width=1400,
            font=dict(size=11),
        )
        
        fig.show()
    
    # Создаем интерактивный слайдер
    interact(
        update_plot,
        n_clusters=IntSlider(
            min=1,
            max=len(X),
            step=1,
            value=OPTIMAL_CLUSTERS,
            description='Кластеры:',
            continuous_update=False
        )
    )


In [48]:
visualize_dendrogram_with_slider(agglomerative, X, data)

interactive(children=(IntSlider(value=4, continuous_update=False, description='Кластеры:', max=300, min=1), Ou…

## Дивизивный метод

In [49]:
divisive = hclust.DivisiveClustering(
    X=X,
    linkage_method=hclust.AverageLinkage(),
    distance_metric=hclust.EuclideanDistance()
)

divisive.fit()

clusters = divisive.predict(OPTIMAL_CLUSTERS)

In [50]:
visualize_clusters(
    data,
    clusters,
    f'Результаты Дивизионной кластеризации (k={OPTIMAL_CLUSTERS})'
)

In [ ]:
visualize_dendrogram_with_slider(divisive, X, data)

TypeError: visualize_dendrogram_with_slider() takes 3 positional arguments but 4 were given

# DBSCAN

In [ ]:
dbscan_model = dbscan.DBSCAN(
    X=X,
    eps=2,
    min_samples=5,  
    distance_metric=dbscan.EuclideanDistance()
)

dbscan_model.fit()

clusters = dbscan_model.get_labels()

In [ ]:
print(f"Количество кластеров: {dbscan_model.get_n_clusters()}")
print(f"Количество шумовых точек: {np.sum(clusters == -1)}")
print(f"Уникальные метки: {np.unique(clusters)}")

Количество кластеров: 5
Количество шумовых точек: 0
Уникальные метки: [0 1 2 3 4]


In [ ]:
visualize_clusters(
    data,
    clusters,
    f'Результаты DBSCAN (eps={dbscan_model.eps}, min_samples={dbscan_model.min_samples})'
)